In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import kagglehub
import glob
import os

path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)


csv_file = glob.glob(f"{path}/*.csv")[0]


df = pd.read_csv(csv_file)


df.head()

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.loc[0]['review']

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

# **Text Preprocessing**

In [ ]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [ ]:
df['review']=df['review'].apply(remove_html_tags)

In [ ]:
df.loc[2]['review']

TextBlob("I thought this was a wonderful way to spend time on a too hot summer weekend sitting in the air conditioned theater and watching a lighthearted comedy The plot is simplistic but the dialogue is witty and the characters are likable even the well bread suspected serial killer While some may be disappointed when they realize this is not Match Point 2 Risk Addiction I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to loveThis was the most Id laughed at one of Woodys comedies in years dare I say a decade While Ive never been impressed with Scarlet Johanson in this she managed to tone down her sexy image and jumped right into a average but spirited young womanThis may not be the crown jewel of his career but it was wittier than Devil Wears Prada and more interesting than Superman a great comedy to go see with friends")

In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [ ]:
df['review']=df['review'].apply(remove_url)

In [ ]:
import string,time
string.punctuation
exclude = string.punctuation


def remove_punc1(text):
  return text.translate(str.maketrans('','',exclude))

In [ ]:
df['review']=df['review'].apply(remove_punc1)

In [ ]:
from textblob import TextBlob

In [ ]:
df['review'] = df['review'].apply(TextBlob)

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# 1. Convert to a set ONCE outside the function
stop_words = set(stopwords.words('english'))

# 2. Optimized function
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word.lower() not in stop_words])

# 3. Apply to DataFrame
df['review'] = df['review'].apply(remove_stopwords)

In [ ]:
df['review'] = df['review'].str.lower()

In [ ]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stem_text(text):
    return " ".join([ps.stem(word) for word in text.split()])

df['cleaned_review'] = df['review'].apply(stem_text)

In [ ]:
df.loc[0]['review']

'one reviewers mentioned watching 1 oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordit called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy high agenda em city home manyaryans muslims gangstas latinos christians italians irish moreso scuffles death stares dodgy dealings shady agreements never far awayi would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high levels graphic violence violence injustice crooked guards wholl sold nickel inmates wholl kill order get away well mannered middle 

In [ ]:
df.loc[0]['cleaned_review']

'one review mention watch 1 oz episod youll hook right exactli happen meth first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc hardcor classic use wordit call oz nicknam given oswald maximum secur state penitentari focus mainli emerald citi experiment section prison cell glass front face inward privaci high agenda em citi home manyaryan muslim gangsta latino christian italian irish moreso scuffl death stare dodgi deal shadi agreement never far awayi would say main appeal show due fact goe show wouldnt dare forget pretti pictur paint mainstream audienc forget charm forget romanceoz doesnt mess around first episod ever saw struck nasti surreal couldnt say readi watch develop tast oz got accustom high level graphic violenc violenc injustic crook guard wholl sold nickel inmat wholl kill order get away well manner middl class inmat turn prison bitch due lack street skill prison experi watch oz may becom c

In [ ]:
total_words = 0
unique_words = set()

for review in df['cleaned_review']:
    tokens = review.split()
    total_words += len(tokens)
    unique_words.update(tokens)

print(f"Total number of words in corpus: {total_words:,}")
print(f"Total number of unique words (Vocabulary): {len(unique_words):,}")

Total number of words in corpus: 5,992,612
Total number of unique words (Vocabulary): 182,633


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
ohe_vect=CountVectorizer(binary=True,max_features=1000)

In [ ]:
ohe_matrix = ohe_vect.fit_transform(df['cleaned_review'])

In [ ]:
print("OHE Matrix Shape:", ohe_matrix.shape)

OHE Matrix Shape: (50000, 1000)


In [ ]:
ohe_df = pd.DataFrame(
    ohe_matrix[:5, :10].toarray(),
    columns=ohe_vect.get_feature_names_out()[:10]
)
ohe_df

,10,20,30,70,80,90,abil,abl,absolut,accent
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


# **problem 4**

In [ ]:
cv = CountVectorizer(max_features=50000)

In [ ]:
bow_matrix = cv.fit_transform(df['cleaned_review'])

In [ ]:
vocabulary = cv.get_feature_names_out()
print(f"Bag of Words Vocabulary Size: {len(vocabulary)}")
print("Sample vocabulary tokens:", vocabulary[:10])

Bag of Words Vocabulary Size: 50000
Sample vocabulary tokens: ['00' '000' '001' '002' '007' '0080' '0083' '01' '010' '02']


In [ ]:
word_counts = bow_matrix.sum(axis=0).A1

In [ ]:
word_freq_df = pd.DataFrame({
    'word': vocabulary,
    'count': word_counts
}).sort_values(by='count', ascending=False).reset_index(drop=True)

In [ ]:
print("\nTop 15 Most Frequent Words:")
print(word_freq_df.head(15))


Top 15 Most Frequent Words:
       word  count
0      movi  98941
1      film  92072
2       one  52654
3      like  43823
4      time  29802
5      good  28899
6      make  28569
7       get  27718
8       see  27572
9   charact  27570
10    watch  27066
11     even  24754
12    stori  24218
13    would  24001
14   realli  22895


# **problem 5**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv_unigram = CountVectorizer(ngram_range=(1, 1))

In [ ]:
cv_unigram.fit(df['cleaned_review'])
vocab_unigram_len = len(cv_unigram.vocabulary_)

In [ ]:
cv_bigram = CountVectorizer(ngram_range=(2, 2))
cv_bigram.fit(df['cleaned_review'])
vocab_bigram_len = len(cv_bigram.vocabulary_)

In [ ]:
cv_trigram = CountVectorizer(ngram_range=(3, 3))
cv_trigram.fit(df['cleaned_review'])
vocab_trigram_len = len(cv_trigram.vocabulary_)

In [ ]:
print(f"Unigram Vocabulary Size : {vocab_unigram_len:,}")
print(f"Bi-gram Vocabulary Size : {vocab_bigram_len:,}")
print(f"Tri-gram Vocabulary Size: {vocab_trigram_len:,}")

Unigram Vocabulary Size : 181,642
Bi-gram Vocabulary Size : 2,827,422
Tri-gram Vocabulary Size: 5,369,731


In [ ]:
print("\nSample Bi-grams :", list(cv_bigram.vocabulary_.keys())[:5])
print("Sample Tri-grams:", list(cv_trigram.vocabulary_.keys())[:5])


Sample Bi-grams : ['one review', 'review mention', 'mention watch', 'watch oz', 'oz episod']
Sample Tri-grams: ['one review mention', 'review mention watch', 'mention watch oz', 'watch oz episod', 'oz episod youll']


# **problem 6**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(max_features=50000)

In [ ]:
tfidf_matrix = tfidf.fit_transform(df['cleaned_review'])

In [ ]:
vocabulary = tfidf.get_feature_names_out()
print(f"TF-IDF Vocabulary Size: {len(vocabulary):,}")
print("First 10 Vocabulary Words:", vocabulary[:10])

TF-IDF Vocabulary Size: 50,000
First 10 Vocabulary Words: ['00' '000' '001' '002' '007' '0080' '0083' '01' '010' '02']


In [ ]:
idf_scores = tfidf.idf_

In [ ]:
idf_df = pd.DataFrame({
    'word': vocabulary,
    'idf_score': idf_scores
})

In [ ]:
print("\nTop 10 Words with LOWEST IDF Scores (appear in many reviews):")
print(idf_df.sort_values(by='idf_score', ascending=True).head(10).reset_index(drop=True))

# Display words with the highest IDF scores (rare, highly specific words)
print("\nTop 10 Words with HIGHEST IDF Scores (appear in very few reviews):")
print(idf_df.sort_values(by='idf_score', ascending=False).head(10).reset_index(drop=True))


Top 10 Words with LOWEST IDF Scores (appear in many reviews):
    word  idf_score
0   movi   1.448246
1   film   1.534899
2    one   1.576380
3   like   1.713492
4   time   1.938710
5   make   1.973993
6   good   1.983787
7    see   1.996979
8  watch   2.025679
9    get   2.036559

Top 10 Words with HIGHEST IDF Scores (appear in very few reviews):
        word  idf_score
0        âme  11.126651
1         är  11.126651
2       zsrr  11.126651
3      iriss  11.126651
4      iritf  11.126651
5     irland  11.126651
6    irmgard  11.126651
7       zinn  11.126651
8     crocki  11.126651
9  pensacola  11.126651
